# PINN 求解氢原子前三个本征态 (1s, 2s, 3s) - 优化版

这个版本引入了以下优化以解决训练结果不佳的问题：
1. **径向重要性采样 (Importance Sampling)**：大部分配点分布在 $r < 4$ 的核心区域。
2. **增强型神经网络**：宽度增加到 128 神经元，训练步数增加到 8000。
3. **体积加权正交损失**：显著提高正交惩罚的有效性。
4. **一维径向剖面图**：直接观察 $\psi(r)$ 的节点（Nodes）结构。

In [ ]:
import torch
import torch.nn as nn
import numpy as np
import matplotlib.pyplot as plt

torch.manual_seed(123)
device = torch.device("cuda" if torch.cuda.is_available() else ("mps" if torch.backends.mps.is_available() else "cpu"))
print(f"Using device: {device}")

In [ ]:
class AtomicNet(nn.Module):
    def __init__(self):
        super(AtomicNet, self).__init__()
        self.net = nn.Sequential(
            nn.Linear(3, 128), nn.Tanh(),
            nn.Linear(128, 128), nn.Tanh(),
            nn.Linear(128, 128), nn.Tanh(),
            nn.Linear(128, 1)
        )
    def forward(self, r_coords):
        return self.net(r_coords)

In [ ]:
def get_sampling_points(N, device):
    # 核心技巧：球坐标采样
    # r 使用指数分布/对数分布，使其在原点附近更密集
    r = 6.0 * torch.rand(N, 1).to(device) ** 2  # 这种幂次分布会让点更多集中在 0 附近
    
    phi = 2 * np.pi * torch.rand(N, 1).to(device)
    costheta = 2 * torch.rand(N, 1).to(device) - 1
    theta = torch.acos(costheta)
    
    x = r * torch.sin(theta) * torch.cos(phi)
    y = r * torch.sin(theta) * torch.sin(phi)
    z = r * torch.cos(theta)
    
    return x.requires_grad_(True), y.requires_grad_(True), z.requires_grad_(True), r

In [ ]:
def calc_pde_loss(model, x, y, z, E):
    psi = model(torch.cat([x, y, z], dim=1))
    
    def get_grad2(f, var):
        g = torch.autograd.grad(f, var, grad_outputs=torch.ones_like(f), create_graph=True)[0]
        return torch.autograd.grad(g, var, grad_outputs=torch.ones_like(g), create_graph=True)[0]

    laplacian = get_grad2(psi, x) + get_grad2(psi, y) + get_grad2(psi, z)
    r = torch.sqrt(x**2 + y**2 + z**2 + 1e-8)
    # PDE Loss
    residual = -0.5 * r * laplacian - psi - E * r * psi
    return torch.mean(residual**2)

In [ ]:
def train_state(n, energy, prev_models=None):
    print(f"\n--- 优化训练中：第 {n} 本征态 (E={energy:.4f}) ---")
    model = AtomicNet().to(device)
    optimizer = torch.optim.Adam(model.parameters(), lr=8e-4)
    scheduler = torch.optim.lr_scheduler.StepLR(optimizer, step_size=4000, gamma=0.5)
    
    epochs = 8000
    for epoch in range(epochs + 1):
        optimizer.zero_grad()
        
        # 每一轮都重新采样，让模型见识更多样化的点
        x_f, y_f, z_f, r_f = get_sampling_points(2000, device)
        
        # 1. PDE Loss
        loss_pde = calc_pde_loss(model, x_f, y_f, z_f, energy)
        
        # 2. Orthogonality Loss (加权惩罚)
        loss_orth = 0
        psi_curr = model(torch.cat([x_f, y_f, z_f], dim=1))
        if prev_models:
            for prev_m in prev_models:
                prev_m.eval()
                with torch.no_grad():
                    psi_prev = prev_m(torch.cat([x_f, y_f, z_f], dim=1))
                # 引入 r^2 权重模拟球积分体积元 dV = r^2 sin(theta) dr dtheta dphi
                loss_orth += torch.mean(psi_curr * psi_prev * (r_f**2 + 0.1))**2 * 1000.0
        
        # 3. 边界与中心约束
        psi_0 = model(torch.zeros(1, 3).to(device))
        loss_ic = torch.mean((psi_0 - 1.0)**2)
        
        total_loss = loss_pde + loss_orth + loss_ic
        total_loss.backward()
        optimizer.step()
        scheduler.step()
        
        if epoch % 2000 == 0:
            print(f"Epoch {epoch} | PDE: {loss_pde.item():.2e} | Orth: {loss_orth if isinstance(loss_orth, int) else loss_orth.item():.2e}")
            
    return model

In [ ]:
models = []
energies = [-0.5, -0.125, -0.0556]
for i, E in enumerate(energies):
    m = train_state(i+1, E, prev_models=models)
    models.append(m)

## 关键验证：一维径向剖面图 (Radial Profile)
验证解析解的核心：
- 1s: 始终大于 0。
- 2s: 在 $r \approx 2$ 处穿过零点（1个节点）。
- 3s: 在 $r \approx 1.9, 7.1$ 等处穿过零点（2个节点）。

In [ ]:
r_eval = torch.linspace(0, 10, 200).view(-1, 1).to(device)
x_eval = r_eval
y_eval = torch.zeros_like(r_eval)
z_eval = torch.zeros_like(r_eval)
coords_eval = torch.cat([x_eval, y_eval, z_eval], dim=1)

plt.figure(figsize=(14, 8))
labels = ["1s Prediction", "2s Prediction", "3s Prediction"]
colors = ['blue', 'red', 'green']

for i, m in enumerate(models):
    m.eval()
    with torch.no_grad():
        psi_val = m(coords_eval).cpu().numpy().flatten()
    plt.plot(r_eval.cpu().numpy(), psi_val, label=labels[i], color=colors[i], linewidth=2)

plt.axhline(0, color='black', linestyle='--')
plt.title("Hydrogen Radial Wavefunctions (\u03c8 vs r)")
plt.xlabel("Radius r (Bohr)")
plt.ylabel("\u03c8(r)")
plt.legend()
plt.grid(True)
plt.show()

In [ ]:
fig = plt.figure(figsize=(24, 8))
titles = ["1s Orbital", "2s Orbital (Node visible)", "3s Orbital (Complex shells)"]
for i in range(3):
    ax = fig.add_subplot(1, 3, i+1, projection='3d')
    
    grid = 50
    l = np.linspace(-8, 8, grid)
    X, Y, Z = np.meshgrid(l, l, l)
    pts = torch.FloatTensor(np.stack([X.flatten(), Y.flatten(), Z.flatten()], axis=1)).to(device)
    
    models[i].eval()
    with torch.no_grad():
        psi = models[i](pts).cpu().numpy().flatten()
    
    density = psi**2
    mask = density > 0.02 * density.max()
    
    ax.scatter(X.flatten()[mask], Y.flatten()[mask], Z.flatten()[mask], 
               c=density[mask], cmap='plasma', alpha=0.2, s=density[mask]*100)
    ax.set_title(titles[i])

plt.tight_layout()
plt.show()